# Light-curve discovery

## Pure ADQL

### For the Future:
The main idea is to use the **Relational Registry** (RR) schema to discover all resources and services that serve timeseries data.
[VODataService 1.3 Working Draft](https://ivoa.net/documents/VODataService/20260324/WD-VODataService-1.3-20260324.html)

Using RegTAP, we will be able to perform global dataset discovery by querying the *productTypeServed* element.
This allows us to ask the Registry precisely whether a table contains timeseries or another product from [Product Type Vocabulary](https://www.ivoa.net/rdf/product-type/2024-05-19/product-type.html)

At the RR level, the following attributes may be useful in the data discovery:
* `rr.capability.standard_id`: identifies the protocol used to access the data (e.g., `ivo://ivoa.net/std/tap%` or `ssap`)
* `rr.resource.res_type`
: by `vs:catalogresource` we could filter data collections metadata, rather than general services (TAP) metadata
* `rr.res_detail` table stores diverse metadata in the `detail_xpath` / `detail_value` pairs (authority, organization, upload limits, etc). The  `detail_xpath='/capability/dataModel/@ivo-id' AND 'detail_value ILIKE '%obscore%'` could help in hunting the ObsCore tables.

### Practical Way I
#### 1. Simple Spectral Access (SSA)
Find resources using SSA protocol:
```sql
SELECT * from rr.res_table
NATURAL JOIN rr.capability
WHERE standard_id = 'ivo://ivoa.net/std/ssa'
  AND ivoid NOT LIKE '%vizier%'
```
Then, try to somehow separate timeseries from spectra. *How???*<br>
*At the individual service level*, for example look at `t_xel` (time axis length), `time_min/time_max/ssa_time_ext` columns -- notable duration may hint at timeseries.<br>
Play around with columns `ucd`?


#### ObsCore
Use RegTap to discover all find all ObsCore tables, then
ask them one by one about dataproduct_type='timeseries'

1. Searching by Table Name (looks so-so):
```sql
SELECT distinct ivoid, table_name
FROM rr.res_table
WHERE table_name ilike '%obscore%'
AND ivoid not like '%vizier%'

Or even this way to get rid of duplications:
```sql
SELECT access_url
FROM rr.res_table
  NATURAL JOIN rr.capability
  NATURAL JOIN rr.interface
  NATURAL JOIN rr.resource
WHERE table_name ILIKE '%obscore%'
  AND standard_id LIKE 'ivo://ivoa.net/std/tap%'
  AND intf_role='std'
  AND ivoid not like '%vizier%'

2. By the `table_utype`
```sql
SELECT DISTINCT table_name, access_url
FROM rr.res_table
  NATURAL JOIN rr.capability
  NATURAL JOIN rr.interface
WHERE
  table_utype ILIKE '%obscore%'
  AND standard_id LIKE 'ivo://ivoa.net/std/tap%'
  AND intf_role='std'
AND ivoid not like '%vizier%'

Following [this Note](https://ivoa.net/documents/Notes/TableReg/20250425/NOTE-TableReg-1.1-20250425.html#tth_sEc4.1), with `res_type='vs:catalogresource'`,
we will be able to extract ObsCore's own coverage (in contrast to the general TAP service that serves it):

```sql
SELECT table_name, access_url, coverage
FROM rr.res_table
  NATURAL JOIN rr.capability
  NATURAL JOIN rr.interface
  NATURAL JOIN rr.resource
  NATURAL JOIN rr.stc_spatial
WHERE
  table_utype LIKE 'ivo://ivoa.net/std/obscore#table-1.%'
  AND standard_id LIKE 'ivo://ivoa.net/std/tap%'
  AND intf_role='std'
  AND res_type='vs:catalogresource'
AND ivoid not like '%vizier%'
```
Still, too few resources have catalogresource type.

3. Look at `dataModel` element<br>
This metadata can be found in `rr.res_detail` table:
```sql
SELECT distinct access_url
FROM rr.res_detail
NATURAL JOIN rr.interface
NATURAL JOIN rr.res_table
WHERE
   (detail_xpath = '/capability/dataModel/@ivo-id'
    AND detail_value ILIKE '%obscore%')
AND ivoid not like '%vizier%'

### The Real Way II (the harsh truth)
#### Find them manually as much as possible. Keep them, update

## Python with PYVO

## References

1. [Discovering Data Collections, 2019 (DDC)](https://www.ivoa.net/documents/discovercollections/20190520/EN-discovercollections-1.1-20190520.html)
2. [VODataService: A VOResource Schema Extension for Describing Collections and Services (Version 1.3)](https://ivoa.net/documents/VODataService/20260324/WD-VODataService-1.3-20260324.html)
3. [VODataService: A VOResource Schema Extension for Describing Collections and Services (Version 1.2)](https://www.ivoa.net/documents/VODataService/20211102/index.html)
4. [TableReg: Registering TAP-Queriable Tables Conforming to Standard Schemas](https://ivoa.net/documents/Notes/TableReg/20250425/NOTE-TableReg-1.1-20250425.html#tth_sEc4.1)
5. [IVOA Documents](https://www.ivoa.net/documents/)
6. [IVOA Vocabulary: Data Product Type](https://www.ivoa.net/rdf/product-type/2024-05-19/product-type.html)
7. [IVOA Registry Relational Schema Version 1.2](https://www.ivoa.net/documents/RegTAP/index.html)
